In [9]:
import torch

from rlaopt.linalg import IdentityConfig, LinSys, NystromConfig
from rlaopt.solvers import PCG, PCGConfig

In [10]:
torch.set_default_dtype(torch.float64)

In [11]:
n = 10000
eigvals = torch.arange(1, n + 1) ** -2.0
reg = 1e-3

U = torch.randn(n, n)
U = torch.linalg.qr(U).Q

A = U @ torch.diag(eigvals) @ U.T
B = torch.randn(n, 10)

In [12]:
lin_sys = LinSys(A, B, reg)
lin_sys.cuda()

LinSys()

In [13]:
preconditioner_config_identity = IdentityConfig()
preconditioner_config_nystrom = NystromConfig(rank=100, base_damping=reg)

In [14]:
solver_config = PCGConfig(
    max_iters=500,
    tol=1e-8,
    preconditioner_config=preconditioner_config_nystrom,
)

In [15]:
solver = PCG(solver_config)
state = solver.init_state(lin_sys)

In [16]:
max_iters = 100

for i in range(max_iters):
    state = solver.step(lin_sys, state)
    print(f"Iteration {i + 1}, residual norm: {state.res_norm}")

Iteration 1, residual norm: tensor([1.6841, 1.5118, 1.6441, 2.3792, 1.5303, 1.8203, 1.5626, 1.5836, 1.5288,
        1.7637], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 2, residual norm: tensor([0.1513, 0.1190, 0.1566, 0.2260, 0.1223, 0.1741, 0.1117, 0.1319, 0.1269,
        0.1229], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 3, residual norm: tensor([0.0095, 0.0093, 0.0086, 0.0141, 0.0084, 0.0128, 0.0076, 0.0092, 0.0096,
        0.0088], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 4, residual norm: tensor([0.0005, 0.0006, 0.0005, 0.0008, 0.0005, 0.0009, 0.0004, 0.0006, 0.0006,
        0.0004], device='cuda:0', grad_fn=<LinalgVectorNormBackward0>)
Iteration 5, residual norm: tensor([2.3816e-05, 3.4156e-05, 2.4381e-05, 4.3882e-05, 2.2741e-05, 5.2825e-05,
        1.7136e-05, 2.9972e-05, 2.6773e-05, 2.2236e-05], device='cuda:0',
       grad_fn=<LinalgVectorNormBackward0>)
Iteration 6, residual norm: tensor([9.8458e-07, 1.6299e-06